# ecDNAInspector_Jaccard_calculations
### This notebook includes all steps needed to go from the final cycle data csv to a csv file with all calculated jaccard indices for each prediction.

In [1]:
import os
import numpy as np
import pandas as pd
import ast

%run 'ecDNAInspector_functions.ipynb'
print("ecDNAInspector functions loaded!")

/Users/spribus/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


ecDNAInspector functions loaded!


### Define required constants and load required data and file paths.

In [3]:
# maximum basepair distance between breakends for them to be considered overlapping
be_overlap_buffer = 100

# replace with the path to your file to store Jaccard calculation data
cycle_jaccard_table_path = 'subset_tool_testing/testing_full_cohort_jacc_data.csv'

# replace with the path to your cycle-level data table (with confidence information)
cycle_level_data_path = 'subset_tool_testing/testing_cycle_data_w_conf.csv'
cohort_df = pd.read_csv(cycle_level_data_path)

### Calculate pairwise Jaccard indices and store in table.

In [4]:
cycle_jaccard_table_data = [["Sample1", "Amp1", "Cycle1", "Sample2", "Amp2", "Cycle2", "BP Jaccard", 
                             "Gene Jaccard", "Overlapping Genes", "Breakend Jaccard", 
                             "Overlapping Breakends"]]

cycle_segment_dict = parse_cycle_data_table_for_cycle_segment_dict(cycle_level_data_path)
cycle_gene_dict = parse_cycle_data_table_for_cycle_genes(cycle_level_data_path)
cycle_be_dict = parse_cycle_data_table_for_cycle_unique_bes(cycle_level_data_path)

samples = list(cycle_segment_dict.keys())
for i in range(len(samples)):
    sample1 = samples[i]
    print("Working on sample: " + str(sample1))
    for amp1 in cycle_segment_dict[sample1]:
        for cycle1 in cycle_segment_dict[sample1][amp1]:
            cycle1_segments = cycle_segment_dict[sample1][amp1][cycle1]
            cycle1_genes = cycle_gene_dict[sample1][amp1][cycle1]
            cycle1_bes = cycle_be_dict[sample1][amp1][cycle1]
            for j in range(i, len(samples)):
                sample2 = samples[j]
                for amp2 in cycle_segment_dict[sample2]:
                    for cycle2 in cycle_segment_dict[sample2][amp2]:
                        if (sample1 == sample2) and (amp1 == amp2) and (cycle1 == cycle2):
                            continue
                        cycle2_segments = cycle_segment_dict[sample2][amp2][cycle2]
                        cycle2_genes = cycle_gene_dict[sample2][amp2][cycle2]
                        cycle2_bes = cycle_be_dict[sample2][amp2][cycle2]
                        bp_jaccard = calc_bp_jaccard(cycle1_segments, cycle2_segments)
                        gene_jaccard, gene_intersect_list = calc_gene_jaccard(cycle1_genes, cycle2_genes)
                        be_jaccard, be_intersections = calc_be_jaccard(cycle1_bes, cycle2_bes, be_overlap_buffer)
                        cycle_jaccard_table_data.append([sample1, amp1, cycle1, sample2, amp2, cycle2, bp_jaccard, 
                                                         gene_jaccard, gene_intersect_list, be_jaccard, be_intersections])
with open(cycle_jaccard_table_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(cycle_jaccard_table_data)

Working on sample: CURTIS_H001731_T01_01_WG01
Working on sample: CURTIS_H000400_T01_01_WG01
Working on sample: CURTIS_H000758_T01_01_WG01
Working on sample: CURTIS_H000741_T01_01_WG01
Working on sample: CURTIS_H000512_T01_01_WG01
Working on sample: CURTIS_H000498_T01_01_WG01
Working on sample: CURTIS_H001487_T01_01_WG01
Working on sample: CURTIS_H000407_T01_01_WG01
Working on sample: CURTIS_H000502_T01_01_WG01
Working on sample: CURTIS_H000736_T01_01_WG01
Working on sample: CURTIS_H000748_T01_01_WG01
Working on sample: CURTIS_H001591_T01_01_WG01
Working on sample: CURTIS_H001700_T01_01_WG01
Working on sample: CURTIS_H000402_T01_01_WG01
Working on sample: CURTIS_H001662_T01_01_WG01
Working on sample: CURTIS_H000723_T01_01_WG01
Working on sample: CURTIS_H001746_T01_01_WG01
Working on sample: CURTIS_H000737_T01_01_WG01
Working on sample: CURTIS_H001526_T01_01_WG01
Working on sample: CURTIS_H001690_T01_01_WG01
Working on sample: CURTIS_H000782_T01_01_WG01
Working on sample: CURTIS_H000558_